# Retrotraducción con más datos sintéticos (R6) — más paráfrasis por oración, dentro de cada fold

Repite la retrotraducción dentro de cada fold, pero con **varias paráfrasis por oración** en vez de una (`--multiplicador`). Cada
paráfrasis adicional se obtiene volviendo a muestrear el paso inglés → español de Helsinki-NLP con otra semilla y traduciéndola al shiwilu con
el NMT de F. Prado. Todo lo hace `validacion_cruzada_en_linea.py`; las paráfrasis repetidas se descartan.

**Estrategia (la misma que el entrenamiento): test / pool / core / dev.** Para cada fold: *test* = el fold (no se toca);
*pool* = los otros 4 folds; el pool se parte en *dev interno* (~1/6) y *core*. Hay dos etapas:
- **Etapa core (elegir `C`):** el aumento se genera/filtra solo con el core y se mide en el dev interno.
- **Etapa pool (modelo final):** el aumento se genera/filtra con todo el pool y se mide en el test.

Así, el dev interno nunca influye en el aumento con el que se elige `C`. En todas las etapas se descartan las filas sintéticas
idénticas a una oración del test.

**Cuánto aumentar.** Hoy Retrotraducción ya aporta ~85% del tamaño del pool real (~480 filas por fold); Mixup, 100%. Con `NIVELES = [2]`
el aporte sube a ~170% (más sintético que real). **No te recomiendo subir más de 2**, y antes de insistir conviene aplicar filtros de calidad a las
paráfrasis: con 1 paráfrasis por oración ya hay ruido (paráfrasis que inventan contenido, duplicados) y Retrotraducción empeora a mBERT de forma
significativa; más volumen del mismo material tiende a amplificar ese ruido. Este cuaderno sirve para comprobarlo.

> **Si Colab se desconecta (guía de reconexión).** No pierdes lo hecho: lo generado se guarda en Drive apenas se produce, y los CSV esenciales de cada nivel
> (predicciones, resumen y volumen) se respaldan en Drive apenas el nivel termina. Para retomar: abre este cuaderno, corre las celdas **0 a 3**, la de
> **Configuración y respaldos** y vuelve a correr la celda **4**: los niveles terminados se restauran en segundos y solo se calcula lo que falta. Después corre
> los pasos **5, 6 y 7** (también restauran desde Drive, así que funcionan en una sesión nueva). El log de cada corrida queda en `Mi unidad/shiwilu_en_linea_logs/`.
> Deja la pestaña abierta mientras corre el paso 4.

**Antes de correr:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **GPU** (T4 sirve).
2. `git commit` + `git push` del repo con los cambios actuales. Esta Colab clona tu repo.
3. Checkpoint en Drive: `Mi unidad/shiwilu_checkpoint/nllb_bidi_lora_v2_1b_loraplus_xl/` (o entrénalo en el paso 3B).
4. Lo generado en la corrida anterior (`shiwilu_en_linea/retrotraduccion/fold<N>_pool.csv`) se **reutiliza**: solo se genera la paráfrasis extra.

**Tiempo:** ~15-30 min por fold y por paráfrasis extra (≈ 1.5-2.5 h para `[2]`), más ~15 min de evaluación. Cada archivo se guarda en Drive apenas se produce.

**Resultado:** un solo `.zip` (descargado y copiado a Drive) con lo generado de todos los niveles y, por cada nivel, las predicciones, el resumen, las comparaciones pareadas,
las curvas ROC (CSV, grilla e imagen por experimento) y el volumen sintético usado.

## 0. Montar Google Drive (aquí queda guardado lo generado, los resultados y los logs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clonar tu repo y el de F. Prado

In [ ]:
import os
GITHUB_USUARIO = "LhiaHC"

%cd /content
!rm -rf /content/shiwilu-tesis
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git clone https://github.com/fapi19/Tesis_Spa-Jeb.git oe2_aumento_de_datos/tecnicas_aumento/tesis_spa_jeb
!git log --oneline -3

ruta = '/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/validacion_cruzada_en_linea.py'
assert os.path.exists(ruta), ('No existe validacion_cruzada_en_linea.py: haz git commit + git push del repo reorganizado '
                              '(carpetas oe2_...) con los cambios nuevos y vuelve a correr esta celda.')
assert 'INTENTOS_RESPUESTA' in open('/content/shiwilu-tesis/oe2_aumento_de_datos/tecnicas_aumento/generate_then_refine.py', encoding='utf-8').read(), (
    'Este clon no tiene la correccion de reintentos de JSON en generate_then_refine.py: haz git push y vuelve a correr esta celda.')
print('OK')

## 2. Entorno aislado de F. Prado (sus versiones fijadas; evita conflictos con los paquetes de Colab)

In [ ]:
%cd /content/shiwilu-tesis/oe2_aumento_de_datos/tecnicas_aumento/tesis_spa_jeb
!pip install -q virtualenv
!rm -rf /content/nmt_venv
!virtualenv -q /content/nmt_venv
!/content/nmt_venv/bin/pip install -q -r requirements/nmt.txt
!/content/nmt_venv/bin/pip install -q sentence-transformers scikit-learn matplotlib pandas
!/content/nmt_venv/bin/python -c "import torch; print('GPU disponible:', torch.cuda.is_available())"
%cd /content/shiwilu-tesis

## 3A. Copiar el checkpoint desde Drive al disco local (recomendado)

In [ ]:
import os
CHECKPOINT = "/content/checkpoint_retro/nllb_bidi_lora_v2_1b_loraplus_xl"

!mkdir -p /content/checkpoint_retro
!cp -r "/content/drive/MyDrive/shiwilu_checkpoint/nllb_bidi_lora_v2_1b_loraplus_xl" /content/checkpoint_retro/
for f in ["adapter_model.safetensors", "adapter_config.json", "tokenizer_config.json"]:
    assert os.path.exists(os.path.join(CHECKPOINT, f)), f"Falta {f} en {CHECKPOINT}."
tam = os.path.getsize(os.path.join(CHECKPOINT, "adapter_model.safetensors")) / 1e9
assert tam > 1.5, f"adapter_model.safetensors pesa solo {tam:.2f} GB (deberia ser ~1.97 GB): copia incompleta."
print(f"OK: checkpoint completo ({tam:.2f} GB) en {CHECKPOINT}")

## 3B. (OPCIONAL) Entrenar el checkpoint desde cero — SOLO si NO lo tienes en Drive

Está protegido: con `ENTRENAR = False` (por defecto) esta celda no hace nada, así que "Ejecutar todo" es seguro.

In [ ]:
ENTRENAR = False   # pon True SOLO si no tienes el checkpoint en Drive (y entonces omite el paso 3A)

if ENTRENAR:
    %cd /content/shiwilu-tesis/oe2_aumento_de_datos/tecnicas_aumento/tesis_spa_jeb
    !/content/nmt_venv/bin/python -m scripts.nmt.30_train_lora --variant xl --rank 32 --alpha 64 --loraplus-lr-ratio 16 --output-dir models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl
    !mkdir -p /content/drive/MyDrive/shiwilu_checkpoint
    !cp -r models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl /content/drive/MyDrive/shiwilu_checkpoint/
    CHECKPOINT = "/content/shiwilu-tesis/oe2_aumento_de_datos/tecnicas_aumento/tesis_spa_jeb/models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl"
    %cd /content/shiwilu-tesis
else:
    print("3B omitido (ENTRENAR = False).")

## Configuración y respaldos (corre esta celda en cada sesión, antes del paso 4)

Aquí eliges los niveles (`NIVELES` = paráfrasis por oración; 1 ya está hecho) y defines las ayudas de respaldo: log persistente en Drive, restauración de niveles ya
evaluados y respaldo de resultados.

In [ ]:
import os, glob, shutil, subprocess, time

NIVELES = [2]   # paráfrasis por oración (1 = la corrida anterior, ya hecha)
PYTHON_NMT = "/content/nmt_venv/bin/python"
PRUEBA = False      # True = prueba rapida: solo el fold 0, con cache, respaldos y etiquetas aparte (no contamina la corrida completa)
REEVALUAR = False   # True = ignora los respaldos de resultados y vuelve a evaluar los niveles (p. ej. si cambiaste codigo o filtros)

SUFIJO = "_prueba" if PRUEBA else ""
CACHE = "/content/drive/MyDrive/shiwilu_en_linea" + SUFIJO                          # lo generado (paráfrasis y NMT de F. Prado)
RESPALDO_RES = "/content/drive/MyDrive/shiwilu_en_linea_resultados" + SUFIJO         # CSV esenciales de cada nivel
RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"
DIR_LOGS = "/content/drive/MyDrive/shiwilu_en_linea_logs"
os.makedirs(RESPALDO_RES, exist_ok=True)
os.makedirs(DIR_LOGS, exist_ok=True)
LOG = f"{DIR_LOGS}/retrotraduccion_{time.strftime('%Y%m%d_%H%M%S')}.txt"
ESENCIALES = ("validacion_cruzada_predicciones_sin_puntuacion_", "validacion_cruzada_resumen_sin_puntuacion_", "volumen_sintetico_sin_puntuacion_")

def tag_de(nivel):
    return f"en_linea_retrotraduccion_colab_x{nivel}{SUFIJO}"

def correr(comando, cwd="/content/shiwilu-tesis"):
    # corre un comando mostrando la salida en vivo y guardandola (sin barras de progreso) en el log de Drive; se detiene si falla
    proc = subprocess.Popen(
        comando, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "MPLBACKEND": "Agg", "TOKENIZERS_PARALLELISM": "false"},
    )
    with open(LOG, "a", encoding="utf-8") as log:
        for linea in proc.stdout:
            print(linea, end="")
            if "Batches:" not in linea and "Loading weights" not in linea:
                log.write(linea)
                log.flush()
    proc.wait()
    assert proc.returncode == 0, f"Fallo (codigo {proc.returncode}): {' '.join(comando)}"

def disponible(tag):
    # True si los CSV esenciales del nivel estan en esta sesion o, si no, en el respaldo de Drive (y entonces los restaura)
    locales = [f"{RES}/{p}{tag}.csv" for p in ESENCIALES]
    if all(os.path.exists(r) for r in locales):
        return True
    copias = [f"{RESPALDO_RES}/{p}{tag}.csv" for p in ESENCIALES]
    if all(os.path.exists(r) for r in copias):
        for r in copias:
            shutil.copy(r, RES)
        return True
    return False

def respaldar(tag):
    # copia a Drive los CSV esenciales del nivel, para no perderlos si Colab se desconecta
    for p in ESENCIALES:
        shutil.copy(f"{RES}/{p}{tag}.csv", RESPALDO_RES)

print("Niveles:", NIVELES, "| PRUEBA:", PRUEBA, "| REEVALUAR:", REEVALUAR)
print("Log de esta sesion:", LOG)

## 4. Generar y evaluar, por nivel de volumen

Para cada nivel y fold: reutiliza las paráfrasis ya guardadas, genera solo las que faltan (y las guarda en Drive), elimina duplicados y copias del test, filtra por etapa,
extrae embeddings y entrena. Al terminar cada nivel, respalda sus CSV esenciales en Drive; si el nivel ya estaba evaluado, lo salta. El paso de traducción **no imprime
progreso**: puede pasar mucho rato sin mostrar nada, y eso no significa que esté colgado.

In [ ]:
assert "CHECKPOINT" in globals(), "Falta correr el paso 3A (o 3B): define la variable CHECKPOINT."

NIVELES_LISTOS = []
for nivel in NIVELES:
    tag = tag_de(nivel)
    print(f"\n=============== Retrotraduccion, {nivel} paráfrasis por oración ===============")
    if not REEVALUAR and disponible(tag):
        print(f"Nivel {nivel}: ya estaba evaluado (en esta sesion o en el respaldo de Drive); se reutiliza sin repetir la evaluacion.")
        NIVELES_LISTOS.append(nivel)
        continue
    try:
        correr([PYTHON_NMT, "oe2_aumento_de_datos/evaluacion/validacion_cruzada_en_linea.py",
                "--tecnica", "retrotraduccion", "--checkpoint", CHECKPOINT, "--multiplicador", str(nivel),
                "--cache", CACHE, "--etiqueta", tag.replace("en_linea_retrotraduccion_", "")] + (["--folds", "0"] if PRUEBA else []))
        respaldar(tag)
        NIVELES_LISTOS.append(nivel)
    except AssertionError as error:
        print(f"\n*** El nivel {nivel} fallo y se omite: {error}\n*** Lo ya generado sigue en Drive: corrige y vuelve a correr esta celda (retoma sola).")
print("\nNiveles terminados (los usan los pasos 5-7):", NIVELES_LISTOS)

## 5. Tablas y curvas ROC (las mismas que el repo), por nivel

In [ ]:
if not PRUEBA:
    import pandas as pd
    from IPython.display import display, Image

    # restaura desde Drive los niveles que no esten en esta sesion (sirve tras una desconexion)
    NIVELES_LISTOS = [n for n in NIVELES if disponible(tag_de(n))]
    TAGS = [tag_de(n) for n in NIVELES_LISTOS]
    print("Niveles disponibles:", NIVELES_LISTOS)

    for TAG in TAGS:
        ENTRADA = f"{RES}/validacion_cruzada_predicciones_sin_puntuacion_{TAG}.csv"
        print("=" * 20, TAG, "=" * 20)
        correr([PYTHON_NMT, "oe2_aumento_de_datos/evaluacion/comparacion_pareada.py", "--entrada", ENTRADA, "--etiqueta", TAG])
        correr([PYTHON_NMT, "oe2_aumento_de_datos/evaluacion/curvas_roc.py", "--entrada", ENTRADA, "--etiqueta", TAG])
        display(Image(f"{RES}/curvas_roc_{TAG}.png"))
        auc_macro = pd.read_csv(f"{RES}/curvas_roc_{TAG}.csv").query("categoria == 'macro'")
        display(auc_macro.pivot(index="modelo", columns="tecnica", values="auc").round(4))

if PRUEBA:
    print('Modo prueba: solo se evaluo el fold 0; las tablas y curvas ROC necesitan los 5 folds.')

## 6. Volumen usado y comparación frente a `sin_aumento` y a la retrotraducción vigente

In [ ]:
if not PRUEBA:
    import sys
    import numpy as np
    sys.path.insert(0, "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion")
    from comparacion_pareada import diferencia_pareada

    P_vig = pd.read_csv(f"{RES}/validacion_cruzada_predicciones_sin_puntuacion.csv")
    vigente = pd.read_csv(f"{RES}/validacion_cruzada_resumen_sin_puntuacion.csv")
    def f1(tabla, tecnica):
        return tabla[tabla.tecnica == tecnica].set_index("modelo").f1_macro_agrupado

    columnas = {"sin aumento": f1(vigente, "sin_aumento"), "retro x vigente (filtro original)": f1(vigente, "retrotraduccion")}
    pcts = {}
    for nivel, TAG in zip(NIVELES_LISTOS, TAGS):
        r = pd.read_csv(f"{RES}/validacion_cruzada_resumen_sin_puntuacion_{TAG}.csv")
        columnas[f"retro x {nivel}"] = f1(r, "retrotraduccion")
        v = pd.read_csv(f"{RES}/volumen_sintetico_sin_puntuacion_{TAG}.csv")
        pcts[nivel] = round(v[v.etapa == "pool"].pct_de_los_reales.mean(), 1)
    display(pd.DataFrame(columnas).round(4))
    print("Volumen sintetico usado en la etapa pool, como % de las oraciones reales del pool:", pcts)

    rng = np.random.default_rng(42)
    filas = []
    for nivel, TAG in zip(NIVELES_LISTOS, TAGS):
        P_nuevo = pd.read_csv(f"{RES}/validacion_cruzada_predicciones_sin_puntuacion_{TAG}.csv")
        P = pd.concat([P_nuevo[P_nuevo.tecnica == "retrotraduccion"].assign(tecnica="nuevo"),
                       P_vig[P_vig.tecnica == "sin_aumento"],
                       P_vig[P_vig.tecnica == "retrotraduccion"].assign(tecnica="vigente")])
        for modelo in ["labse", "mbert", "xlmr"]:
            for a, b in [("nuevo", "sin_aumento"), ("nuevo", "vigente")]:
                d, lo, hi = diferencia_pareada(P, (modelo, a), (modelo, b), rng)
                filas.append({"nivel": nivel, "modelo": modelo, "comparacion": f"{a} - {b}", "diferencia_f1": round(d, 4),
                              "ic95_bajo": round(lo, 4), "ic95_alto": round(hi, 4), "distinguible_de_cero": not (lo <= 0 <= hi)})
    display(pd.DataFrame(filas))

## 7. Descargar el resultado (un solo `.zip`)

Contiene `en_linea/retrotraduccion/` (lo generado, de todos los niveles) y `resultados/` (por nivel: predicciones y resumen con los 12 experimentos, comparaciones
pareadas, curvas ROC en CSV, grilla e imágenes por experimento, y el volumen sintético). Se descarga y además se copia a `Mi unidad/` por si la descarga falla.

In [ ]:
from google.colab import files

NIVELES_LISTOS = [n for n in NIVELES if disponible(tag_de(n))]
carpeta = "/content/salida_retro_en_linea"
shutil.rmtree(carpeta, ignore_errors=True)
os.makedirs(carpeta + "/resultados"); os.makedirs(carpeta + "/en_linea")
shutil.copytree(f"{CACHE}/retrotraduccion", carpeta + "/en_linea/retrotraduccion")        # lo generado, todos los niveles
for ruta_ in glob.glob(f"{RES}/*en_linea_retrotraduccion_colab_x*"):                                   # CSV, PNG y carpetas de curvas ROC de todos los niveles
    destino = carpeta + "/resultados/" + os.path.basename(ruta_)
    shutil.copytree(ruta_, destino) if os.path.isdir(ruta_) else shutil.copy(ruta_, destino)
zip_path = shutil.make_archive("/content/salida_retro_en_linea", "zip", carpeta)
zip_drive = shutil.copy(zip_path, "/content/drive/MyDrive/salida_retro_en_linea" + SUFIJO + ".zip")   # copia de seguridad en Drive
print("Niveles incluidos:", NIVELES_LISTOS)
print("Copia del .zip en Drive:", zip_drive)
print(sorted(os.listdir(carpeta + "/resultados")))
try:
    files.download(zip_path)
except Exception as error:
    print("La descarga automatica fallo (", error, "). Descarga el .zip desde Drive:", zip_drive)